# BrainDecoding — Google Colab GPU 留出测试集 (`test`) 评估、负对照与可视化工作流

本 Notebook 专门用于在 **Google Colab GPU 实例** 上对已完成训练的模型执行**留出测试集 (`split=test`) 全面评估、时间错位/全零信号负对照检验与可视化对比**：
- **支持三阶段完整横向对比（`--all-stages`）**：
  - **阶段 1 (`word`)**：词级脑编码器（纯 CNN，`use_transformer: false`）
  - **阶段 2.1 (`context`)**：词级脑编码器 + 16 层全局上下文 Transformer（`use_transformer: true`，完整窗口，无局部滚动修订）
  - **阶段 2.2 (`revision`)**：词级脑编码器 + 16 层局部窗口滚动修订（`use_transformer: true`，`K0/K1/K2` 局部窗口滚动回修）
- **支持独立解耦的负对照配置体系（`configs/controls/`）**：
  - **时间错位真实信号负对照（Temporal Shift Control）**：在同受试者、同连续录音内对真实脑磁/脑电信号施加纯历史因果错位 $\delta \in \{-15\text{ s}, -30\text{ s}, -45\text{ s}, -60\text{ s}, -90\text{ s}, -120\text{ s}\}$（`[-15, -30, -45, -60, -90, -120]` 秒）确定性圆周移位，配置文件位于 `configs/controls/temporal_shift/`；
  - **留一法共模扣除（Leave-One-Out Common-Mode Subtraction）**：在单次前向推理完 1 个 Clean (0s) 与 6 个 Shifted 信号后，对 Clean (0s) 扣除全部 6 个 Shifted 均值（权重 $\eta \in \{0.5, 0.8, 1.0\}$），并对每个 Shifted 条件公平扣除其余 5 个 Shifted 均值，配置文件位于 `configs/controls/common_mode_subtraction/`；
  - **全零神经信号负对照（Zero-Signal Control）**：将输入脑信号置零以检验无脑信号时的机遇基线，配置文件位于 `configs/controls/zero_signal/`；
- **支持动态时间窗长度（`WINDOW_SECONDS`）**：默认自动继承已训练检查点保存的窗口长度（如 `1.0s`、`1.25s`、`1.5s`），也可显式覆写；
- **支持 `EXP_TAG` 与 `WORD_EXP_TAG` 跨实验标签组合评估**：按 `runs/<dataset>/<EXP_TAG>/<stage>/...` 自动读取对应超参实验的权重，且支持通过 `WORD_EXP_TAG` 引用不同实验标签下的阶段 1 (`word`) 参与三阶段对比；
- **自动输出多维度定量报表与高清可视化图表**：
  1. 多词表规模检索曲线（`N=20, 50, 100, 150, Full Vocab` 的 Macro/Micro Top-10、Top-1、MRR、中位秩 vs 随机基线）；
  2. 真实对齐脑信号（`Clean 0s`）对比 $-15\text{ s}, -30\text{ s}, -45\text{ s}, -60\text{ s}, -90\text{ s}, -120\text{ s}$ 错位脑信号负对照的检索准确率与降幅对比图（`negative_control_test.png` 与 `stage_control_comparison_test.png`）；
  3. 分受试者（`sub-01` .. `sub-N`）测试集 50 词检索准确率拆解；
  4. 阶段 2.2 滚动修订各延迟视野（`K0 -> K1 -> K2`）测试集增益拆解。

In [ ]:
# 1. 检查 GPU 并挂载 Google Drive、配置环境变量
!nvidia-smi

import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('非 Colab 环境，跳过 drive.mount')

# ==================== 用户可修改参数 ====================
GIT_REPO_URL = 'https://github.com/wdongdong0708-jpg/BrainDecoding2.git'
GIT_BRANCH = 'master'
DRIVE_BASE_DIR = '/content/drive/MyDrive/BrainDecoding'

# 选择要评估的数据集或单阶段 YAML 配置文件：
#   - 'configs/training/pallier2025_word_encoder.yaml'                              (法语 MEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/pallier2025_context_transformer.yaml'                       (法语 MEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/pallier2025_rolling_revision.yaml'                          (法语 MEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
#   - 'configs/training/siamese_debiased/pallier2025_siamese_debiased_revision.yaml' (法语 MEG 阶段 2.2: 孪生共模抵消 + 错位零化正则 + 预提取极速训练)
#   - 'configs/training/siamese_debiased/pallier2025_siamese_debiased_context.yaml'  (法语 MEG 阶段 2.1: 孪生共模抵消 + 错位零化正则 + 预提取极速训练)
#   - 'configs/training/chineseeeg2_word_encoder.yaml'                              (中文 EEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/chineseeeg2_context_transformer.yaml'                       (中文 EEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/chineseeeg2_rolling_revision.yaml'                          (中文 EEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
#   - 'configs/training/siamese_debiased/chineseeeg2_siamese_debiased_revision.yaml' (中文 EEG 阶段 2.2: 孪生共模抵消 + 错位零化正则 + 预提取极速训练)
DATASET = 'pallier2025'               # 'pallier2025' 或 'chineseeeg2_littleprince'
CONFIG_FILE = 'configs/training/siamese_debiased/pallier2025_siamese_debiased_revision.yaml'
MODALITY = 'meg'                      # 'meg' (pallier2025) 或 'eeg' (chineseeeg2)

# 选择独立解耦的负对照配置文件（位于 configs/controls/，与 configs/word_decoding/ 解耦）：
#   - 'configs/controls/temporal_shift/temporal_shift_control.yaml'                      (通用 -15s, -30s, -45s, -60s, -90s, -120s 纯历史因果时间错位负对照)
#   - 'configs/controls/common_mode_subtraction/common_mode_subtraction_control.yaml'    (探索性留一法共模扣除: eta in {0.5, 0.8, 1.0} + 纯历史因果时间错位负对照)
#   - 'configs/controls/temporal_shift/pallier2025_temporal_shift_control.yaml'           (Pallier2025 专属 -15s, -30s, -45s, -60s, -90s, -120s 纯历史因果时间错位负对照)
#   - 'configs/controls/temporal_shift/chineseeeg2_temporal_shift_control.yaml'           (ChineseEEG-2 专属 -15s, -30s, -45s, -60s, -90s, -120s 纯历史因果时间错位负对照)
#   - 'configs/controls/zero_signal/zero_signal_control.yaml'                            (全零脑磁/脑电信号负对照)
#   - ''                                                                                 (留空 '' 则仅运行标准 Clean 0s 评估)
CONTROL_CONFIG = 'configs/controls/common_mode_subtraction/common_mode_subtraction_control.yaml'
SHIFT_SECONDS = '-15,-30,-45,-60,-90,-120'  # 纯历史因果时间错位窗口网格（秒；显式传参确保不依赖旧配置缓存，填 None 则从 YAML 读取）

EXP_TAG = 'lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window'              # 待评估的超参实验标签（定位 runs/<dataset>/<EXP_TAG>/<stage>/...；留空 '' 则为 runs/<dataset>/<stage>/...）
WORD_EXP_TAG = 'lr4e-4_bs256'                     # 若阶段 1 (word) 保存在另一个实验标签下，在此填入该标签名（留空 '' 默认与 EXP_TAG 相同；填 'default' 定位无标签根目录 word）
WINDOW_SECONDS = 1                 # 单字脑信号时间窗长度（秒，如 0.5, 0.75, 1.0, 1.25, 1.5, 2.0；填 None 则自动沿用检查点训练时的窗口长度）
WINDOW_START_OFFSET_SECONDS = None    # 相对于词起始时刻的窗口起点偏移（秒；填 None 则自动沿用检查点配置）
SEED = 0                              # 随机种子: 0, 1, 2
SUBSET_ARGS = '--subject all'         # 必须与训练时的受试者子集保持一致
RUN_WORD_LEVEL_EVAL = False           # 默认 False：词级评估已完成时直接跳过 Step 4(16.4GB解包)/Step 5/Step 6，0秒直达 Step 7 S3 序列解码！如需重跑词级评估改为 True

# Stage 3 (S3) 序列解码协议参数（供下方 Step 7 序列解码使用）：
#   - Colab L4 (22.5GB) / A100 (40GB+) 推荐使用 'Qwen/Qwen3-8B-Base'
#   - Colab T4 (15GB) 显存较紧时可切换为 'Qwen/Qwen3-4B-Base'
S3_LANGUAGE_MODEL = 'Qwen/Qwen3-8B-Base'
S3_TOP_K = 20                         # 闭集词表神经检索 Top-K 候选截断数（局部 Top-20 Softmax 归一化）
S3_BEAM_WIDTH = 5                     # 方案一 K2 定稿主干束宽 B
S3_CMS_ETA = 0.0                      # 孪生去偏检查点推荐设为 0.0（或 0.25），保留 61.01% Clean 最高神经信噪比
S3_CONTROL_SHIFT = -75.0              # 训练未见的纯历史因果错位负对照秒数（-75s）
S3_K_OBSERVATIONS = '1,2,5,10'        # 跨被试多观测聚合曲线 k ∈ {1, 2, 5, 10}（对标 d'Ascoli 2024 & SimpleB2T 2026）
S3_VOCAB_SIZES = '50,100,250,500,1000,2020'  # 零训练成本多尺度候选词表扫描 |V|（按训练集词频降序截断；|V|=2020 自动零开销复用已有全词库结果）
S3_CORE_VOCAB_SIZE = 250              # 对标 SimpleB2T (2026) 与 Jayalath (2025) 的高覆盖核心词表规模 |V|=250（同时输出口径1全集与口径2 100%/80%/70% 高覆盖核心子集）
S3_MAX_SEGMENTS = None                # 填 None 跑全量验证/测试集；填整数（如 20）可做极速联调冒烟
# ========================================================

CONTROL_ARGS = f'--control-config {CONTROL_CONFIG}' if str(CONTROL_CONFIG).strip() else ''
if str(CONTROL_CONFIG).strip() and 'zero_signal' not in str(CONTROL_CONFIG).lower() and SHIFT_SECONDS is not None and str(SHIFT_SECONDS).strip() != '':
    CONTROL_ARGS = f'{CONTROL_ARGS} --shift-seconds {SHIFT_SECONDS}'.strip()
WINDOW_ARGS = f'--window-seconds {WINDOW_SECONDS}' if WINDOW_SECONDS is not None and str(WINDOW_SECONDS).strip() != '' else ''
if WINDOW_START_OFFSET_SECONDS is not None and str(WINDOW_START_OFFSET_SECONDS).strip() != '':
    WINDOW_ARGS = f'{WINDOW_ARGS} --window-start-offset-seconds {WINDOW_START_OFFSET_SECONDS}'.strip()
S3_MAX_SEG_ARGS = f'--max-segments {S3_MAX_SEGMENTS}' if S3_MAX_SEGMENTS is not None else ''

CODE_DIR = '/content/code'
VENV_DIR = '/content/proj-venv'
PYTHON = f'{VENV_DIR}/bin/python'
LINKS_DIR = '/content/studies_links'
LOCAL_CACHE_DIR = '/content/local_cache'

DRIVE_DATA_DIR = f'{DRIVE_BASE_DIR}/data'
DRIVE_ARCHIVE_DIR = f'{DRIVE_BASE_DIR}/archives'
DRIVE_RESULTS_DIR = f'{DRIVE_BASE_DIR}/runs'
DRIVE_MODELS_DIR = f'{DRIVE_BASE_DIR}/models'

for folder in (
    LINKS_DIR,
    LOCAL_CACHE_DIR,
    DRIVE_DATA_DIR,
    DRIVE_ARCHIVE_DIR,
    DRIVE_RESULTS_DIR,
    DRIVE_MODELS_DIR,
):
    Path(folder).mkdir(parents=True, exist_ok=True)

os.environ['BRAINDECODING_DATA_DIR'] = LINKS_DIR
os.environ['BRAINDATA_ROOT'] = LINKS_DIR
os.environ['BRAINDECODING_CACHE_DIR'] = LOCAL_CACHE_DIR
os.environ['BRAINDECODING_RESULTS_DIR'] = DRIVE_RESULTS_DIR
os.environ['BRAINDECODING_EXP_TAG'] = EXP_TAG
os.environ['BRAINDECODING_WORD_EXP_TAG'] = WORD_EXP_TAG
os.environ['BRAINDECODING_MODEL_DIR'] = DRIVE_MODELS_DIR

print('DATASET                   =', DATASET)
print('CONFIG_FILE               =', CONFIG_FILE)
print('CONTROL_CONFIG            =', CONTROL_CONFIG or '(disabled - clean only)')
print('SHIFT_SECONDS             =', SHIFT_SECONDS if (str(CONTROL_CONFIG).strip() and SHIFT_SECONDS) else '(from YAML / disabled)')
print('WINDOW_SECONDS            =', WINDOW_SECONDS if WINDOW_SECONDS is not None else '(auto from checkpoint / YAML)')
print('BRAINDECODING_RESULTS_DIR =', os.environ['BRAINDECODING_RESULTS_DIR'])
print('BRAINDECODING_EXP_TAG     =', os.environ['BRAINDECODING_EXP_TAG'] or '(default)')
print('BRAINDECODING_WORD_EXP_TAG=', os.environ['BRAINDECODING_WORD_EXP_TAG'] or '(same as EXP_TAG / fallback default)')

In [ ]:
# 2. 克隆或同步最新代码到 /content/code
!if [ ! -d "{CODE_DIR}/.git" ]; then \
    git clone -b {GIT_BRANCH} {GIT_REPO_URL} {CODE_DIR}; \
else \
    git -C {CODE_DIR} fetch origin && git -C {CODE_DIR} checkout {GIT_BRANCH} && git -C {CODE_DIR} pull --ff-only; \
fi

In [ ]:
# 3. 初始化虚拟环境 + 建立 /content/studies_links 软链接桥
!python3 -u {CODE_DIR}/scripts/colab_setup.py \
    --venv-dir {VENV_DIR} \
    --project-root {CODE_DIR} \
    --drive-data-dir {DRIVE_DATA_DIR} \
    --links-dir {LINKS_DIR} \
    --local-cache-dir {LOCAL_CACHE_DIR}

In [ ]:
# 4. 秒级解包 Google Drive 上的 .tar 缓存归档回 /content/local_cache
#    若 RUN_WORD_LEVEL_EVAL=False 且云盘已导出 neural_vocabulary_scores_test.pt，则直接 0 秒跳过 16.4GB 解包！
_tag_part = EXP_TAG.strip() if (EXP_TAG and EXP_TAG.strip().lower() not in ('', 'default', 'root', 'none')) else ''
_s3_dir = Path(DRIVE_RESULTS_DIR) / DATASET / _tag_part / 'sequence_decoding' / f'seed-{int(SEED):03d}' if _tag_part else Path(DRIVE_RESULTS_DIR) / DATASET / 'sequence_decoding' / f'seed-{int(SEED):03d}'
_exported_test_pt = _s3_dir / 'neural_vocabulary_scores_test.pt'
if (not RUN_WORD_LEVEL_EVAL) and _exported_test_pt.is_file():
    print(f'[0秒跳过解包] 已检测到云盘存在 {_exported_test_pt} 且 RUN_WORD_LEVEL_EVAL=False，S3 多尺度词表解码直接读取 .pt 矩阵，无需解包 16.4GB 原始脑信号缓存！')
else:
    !cd {CODE_DIR} && {PYTHON} -u scripts/cache_archive.py unpack \
        --archive-dir {DRIVE_ARCHIVE_DIR} \
        --cache-dir {LOCAL_CACHE_DIR} \
        --modality {MODALITY} \
        --skip-existing

In [ ]:
# 5. 【可选：RUN_WORD_LEVEL_EVAL=True 时执行】一键评估当前数据集与 EXP_TAG 下所有已训练阶段（word / context / revision）
if RUN_WORD_LEVEL_EVAL:
    !cd {CODE_DIR} && {PYTHON} -u -m braindecoding eval-run \
        --dataset {DATASET} \
        --seed {SEED} \
        --split test \
        --all-stages \
        {WINDOW_ARGS} \
        {CONTROL_ARGS} \
        {SUBSET_ARGS}
else:
    print('[跳过词级重跑] RUN_WORD_LEVEL_EVAL=False，直接复用云盘已完成的词级评估结果（如需重跑词级评估请将 Cell 1 中 RUN_WORD_LEVEL_EVAL 设为 True）。')

from IPython.display import Image, display

test_pngs = sorted(
    Path(DRIVE_RESULTS_DIR).rglob(f'seed-{int(SEED):03d}/figures/*test*.png'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
allowed_tags = {t for t in (EXP_TAG, WORD_EXP_TAG) if t and t.lower() not in ('default', 'root', 'none')}
shown = set()
for png_path in test_pngs:
    if allowed_tags and not any(t in png_path.parts for t in allowed_tags):
        continue
    key = (png_path.parent.parent.parent.name, png_path.name)
    if key in shown:
        continue
    shown.add(key)
    print(f'\n[测试集图表] {png_path}')
    display(Image(filename=str(png_path)))

In [ ]:
# 6. 【可选：RUN_WORD_LEVEL_EVAL=True 时执行】仅评估单个指定的 CONFIG_FILE 在测试集 (test) 上的表现
if RUN_WORD_LEVEL_EVAL:
    !cd {CODE_DIR} && {PYTHON} -u -m braindecoding eval-run \
        --config {CONFIG_FILE} \
        --seed {SEED} \
        --split test \
        {WINDOW_ARGS} \
        {CONTROL_ARGS} \
        {SUBSET_ARGS}
else:
    print('[跳过单阶段词级重跑] RUN_WORD_LEVEL_EVAL=False，跳过重复评估。')

In [ ]:
# 7. 【Stage 3 / S3 序列解码 + 多尺度候选词表 (|V|=50..2020) + 高覆盖核心集 (|V|=250)】
#    - 自动检测并 100% 复用云盘已有的 neural_vocabulary_scores_{val,test}.pt、state_hyperparameter_calibration.json
#      以及 sequence_decoding_test.json 中已跑完的 |V|=2020 全词库主条件与 k=1..10 跨被试聚合结果（0秒跳过不重跑！）
#    - 增量补跑截断词表尺度 |V| ∈ {250, 50, 100, 500, 1000}（按训练集词频严格降序截断，每跑完一个尺度立即落盘保存）
#    - 自动输出：全词库主表 + 高覆盖核心词表 |V|=250 平行主表（口径1全集 + 口径2 100%/80%/70% 核心子集）+ 多尺度词表缩放曲线图
!cd {CODE_DIR} && {PYTHON} -u -m braindecoding decode-sentences \
    --step all \
    --dataset {DATASET} \
    --split test \
    --seed {SEED} \
    --language-model {S3_LANGUAGE_MODEL} \
    --top-k {S3_TOP_K} \
    --beam-width {S3_BEAM_WIDTH} \
    --cms-eta {S3_CMS_ETA} \
    --control-shift-seconds {S3_CONTROL_SHIFT} \
    --k-observations {S3_K_OBSERVATIONS} \
    --vocab-sizes {S3_VOCAB_SIZES} \
    --core-vocab-size {S3_CORE_VOCAB_SIZE} \
    {WINDOW_ARGS} \
    {S3_MAX_SEG_ARGS} \
    {SUBSET_ARGS}

from IPython.display import Image, display

s3_pngs = sorted(
    Path(DRIVE_RESULTS_DIR).rglob(f'sequence_decoding/seed-{int(SEED):03d}/evaluation_test/*.png'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
for png_path in s3_pngs:
    if EXP_TAG and EXP_TAG not in ('default', 'root', 'none') and EXP_TAG not in png_path.parts:
        continue
    print(f'\n[S3 序列解码图表] {png_path}')
    display(Image(filename=str(png_path)))

In [ ]:
# 最后一个单元格
from google.colab import runtime
print("全部训练任务已完成，释放运行时以停止扣费。")
runtime.unassign()